# Session 7 — Sourcing data I: pull vs push, SFTP, S3 and databases

Every pipeline starts with a question that sounds simple: **how do we get the data?**
So far our files were just "there" in a folder. In real life, data sits on partners' servers, in cloud storage,
in production databases, behind APIs — and it arrives in very different ways.

**Today you will learn**
1. The ways data reaches a data platform: **pull vs push**, batch vs streaming
2. Handling **credentials** safely
3. **SFTP**: download files from a server, only the new ones
4. **Object storage (S3)**: buckets, keys and prefixes — with a real public AWS bucket
5. **Databases**: read from a production database without hurting it, incrementally
6. **Capstone kickoff**

Next session we cover the fourth big source type: **APIs and JSON**.

## Homework review (15 min)
Open `homework/hw06_pipeline.ipynb`.

---
## Part 1 — Where does data come from?

| Source | Wavely example | Typical way in |
|---|---|---|
| Files from partners | a record label sends its new releases every night | **SFTP** server, shared S3 bucket |
| Cloud object storage | our app uploads event files every hour | **S3** / Azure Blob / Google Cloud Storage |
| Production databases | subscriptions and payments live in PostgreSQL | read replica, JDBC, **change data capture** |
| APIs | artist popularity from Deezer | HTTP + JSON (session 8) |
| Event streams | every play, the moment it happens | Kafka, Kinesis, Pub/Sub (streaming) |

**Pull vs push**

| | **Pull** — we go and get it | **Push** — they send it to us |
|---|---|---|
| examples | download from SFTP, list an S3 bucket, query a database, call an API | partner uploads to *our* S3 bucket, webhooks, message queues |
| who decides when | we do (schedule) | the sender |
| we must handle | "what is new since last time?" | data arriving at any time, maybe twice |

Both end the same way: the data is copied **unchanged** into our **landing zone / bronze** layer first (session 6).

**Before connecting to any new source, ask:** What format? How often, how much? How do we authenticate?
How do we know what is new? What happens if a delivery is late, broken or sent twice? Who do we call when it breaks?
Written down and agreed with the provider, this is called a **data contract**.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

In [ ]:
# Extra libraries for today: paramiko (SFTP) and boto3 (AWS S3)
for package in ["paramiko", "boto3"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)

import fnmatch
import os
import shutil

LANDING = DATA.parent / "landing"
shutil.rmtree(LANDING, ignore_errors=True)  # fresh start for the lesson
print("Landing zone:", LANDING)

---
## Part 2 — Secrets: never put passwords in code

Every real source needs credentials: passwords, SSH keys, access keys, tokens. Rules:

- **Never** write them in a notebook or script. Code ends up on GitHub, in screenshots, in emails.
- Read them from the **environment** (`os.environ`), a **secrets manager** (AWS Secrets Manager, Azure Key Vault),
  or in Colab from the *Secrets* panel (key icon on the left: `from google.colab import userdata; userdata.get("SFTP_PASSWORD")`).
  For interactive use, `getpass.getpass()` asks for a password without showing it.
- Use **least privilege**: a read-only user that can only see the folder it needs.

In [ ]:
# The public test server below publishes its demo login; real credentials come from the environment or a secrets store.
SFTP_HOST = "test.rebex.net"
SFTP_USER = os.environ.get("SFTP_USER", "demo")
SFTP_PASSWORD = os.environ.get("SFTP_PASSWORD", "password")

---
## Part 3 — SFTP: files on a server

**SFTP** (SSH File Transfer Protocol) is how countless companies still exchange files: banks, retailers, record labels.
You connect to a server (port 22) with a user name and a password or an **SSH key**, then list, download and upload files.
Python's `paramiko` library speaks SFTP.

Let's connect to a **real public test server** that anyone can read from:

In [ ]:
import paramiko


def connect_sftp(host, user, password):
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    # Accepting unknown host keys is only OK for this public test server; in production, pin the server's key.
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(host, port=22, username=user, password=password, timeout=10, banner_timeout=10, auth_timeout=10)
    return ssh, ssh.open_sftp()


try:
    ssh, sftp = connect_sftp(SFTP_HOST, SFTP_USER, SFTP_PASSWORD)
    for entry in sftp.listdir_attr("/"):
        print(f"{entry.filename:<20} {entry.st_size:>8} bytes")
    with sftp.open("/readme.txt") as f:
        print(f.read().decode()[:300])
    sftp.close()
    ssh.close()
except (OSError, paramiko.SSHException) as error:
    print(f"Could not reach {SFTP_HOST} ({type(error).__name__}). Port 22 is often blocked on company networks - that's OK.")

> **Under the hood:** the first time you connect, the server shows its **host key** (a fingerprint). Checking it protects you from
> someone pretending to be the server (a "man-in-the-middle" attack). In production you store the expected key and reject anything else.

### Wavely's partner: Danube Records
Danube Records drops one file per day with its upcoming releases on its SFTP server, in the folder `/danube_records`.
We can't upload files to the public test server, so we **simulate** the partner's server with a small class that behaves exactly like
paramiko's SFTP client (same methods: `listdir_attr`, `get`) but reads a local folder. Your code will not notice the difference —
and it would work unchanged against the real server.

In [ ]:
from types import SimpleNamespace


class LocalSFTP:
    """Same methods as paramiko's SFTPClient, backed by a local folder."""

    def __init__(self, root):
        self.root = Path(root)

    def listdir_attr(self, path="."):
        folder = self.root / path.strip("/")
        return [SimpleNamespace(filename=p.name, st_size=p.stat().st_size, st_mtime=p.stat().st_mtime) for p in sorted(folder.iterdir())]

    def get(self, remotepath, localpath):
        shutil.copy(self.root / remotepath.strip("/"), localpath)


partner_sftp = LocalSFTP(DATA / "sftp_drop")
for entry in partner_sftp.listdir_attr("/danube_records"):
    print(f"{entry.filename:<30} {entry.st_size:>6} bytes")

Notice two traps:
- `README.txt` is not data.
- `releases_2026-09-06.csv.tmp` is a file **still being uploaded**. Partners upload as `.tmp` and rename when finished.
  Reading a half-written file is a classic bug!

### Exercise 1 (together) — `pull_new_files`
We go through this function together. It downloads only the files we need:
1. list the remote folder with `sftp.listdir_attr(remote_dir)`
2. keep only names matching `pattern` (`fnmatch.fnmatch(name, pattern)`, e.g. pattern `"releases_*.csv"`)
3. skip files that already exist in `local_dir` **with the same size** (already downloaded)
4. download the rest with `sftp.get(remote_path, local_path)` (paths as strings; remote paths use `/`)
5. return the list of downloaded file names

In [ ]:
def pull_new_files(sftp, remote_dir, local_dir, pattern):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []
    for entry in sftp.listdir_attr(remote_dir):  # 1. list the remote folder
        if not fnmatch.fnmatch(entry.filename, pattern):  # 2. skip README.txt, *.tmp, ...
            continue
        target = local_dir / entry.filename
        if target.exists() and target.stat().st_size == entry.st_size:  # 3. already downloaded
            continue
        sftp.get(f"{remote_dir}/{entry.filename}", str(target))  # 4. download
        downloaded.append(entry.filename)
    return downloaded  # 5.


print(pull_new_files(partner_sftp, "/danube_records", LANDING / "danube_records", "releases_*.csv"))  # expected: 5 files
print(pull_new_files(partner_sftp, "/danube_records", LANDING / "danube_records", "releases_*.csv"))  # expected: []

The second run downloads nothing: our pull is **idempotent**. Now the landed files go into Spark like any other file:

### Exercise 2 (Core) — What is Danube Records releasing?
1. Read all landed `releases_*.csv` files into `releases` (header, `inferSchema=True`) and add `_source_file`.
2. Show the number of releases and the total number of tracks per `artist_name` and `release_type`.
3. Which releases come out in the week of **21–27 September 2026**?

In [ ]:
# YOUR CODE HERE

**Stretch:** if the real test server was reachable, call `pull_new_files` on it (folder `/pub/example`, pattern `"*.txt"`)
using a fresh `connect_sftp(...)`. Same function, real server.

---
## Part 4 — Object storage: S3

Cloud **object storage** is the foundation of nearly every modern data platform: Amazon **S3**, Azure Blob/Data Lake Storage, Google Cloud Storage.

- A **bucket** is a top-level container, e.g. `noaa-gsod-pds`.
- Every file is an **object** with a **key**, e.g. `2024/11036099999.csv`. There are no real folders: `2024/` is just a **prefix** of the key.
- Access is controlled with credentials or, inside the cloud, with **IAM roles** (no passwords at all — the best option).
- Many organisations publish **open data** in public buckets that anyone can read anonymously.

We'll read real weather data from the **NOAA Global Surface Summary of the Day** in the public AWS bucket `noaa-gsod-pds`:
one CSV per weather station per year. Below are the main airports of our six countries.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError

BUCKET = "noaa-gsod-pds"
STATIONS = {"11036099999": "AT", "11816099999": "SK", "12839099999": "HU",
            "11518099999": "CZ", "10385099999": "DE", "12375099999": "PL"}


class LocalS3:
    """Same methods as a boto3 S3 client (list_objects_v2, download_file), backed by data/s3_fallback."""

    def __init__(self, root):
        self.root = Path(root)

    def list_objects_v2(self, Bucket, Prefix="", **kwargs):
        base = self.root / Bucket
        keys = sorted(p.relative_to(base).as_posix() for p in base.rglob("*") if p.is_file())
        contents = [{"Key": k, "Size": (base / k).stat().st_size} for k in keys if k.startswith(Prefix)]
        return {"Contents": contents, "KeyCount": len(contents), "IsTruncated": False}

    def download_file(self, Bucket, Key, Filename):
        shutil.copy(self.root / Bucket / Key, Filename)


def connect_s3():
    # UNSIGNED = anonymous access, only possible for public buckets. Real buckets use credentials or an IAM role.
    s3 = boto3.client("s3", region_name="us-east-1",
                      config=Config(signature_version=UNSIGNED, connect_timeout=5, retries={"max_attempts": 1}))
    try:
        s3.list_objects_v2(Bucket=BUCKET, Prefix="2024/11036", MaxKeys=1)
        print("Connected to the real public S3 bucket")
        return s3
    except (BotoCoreError, ClientError) as error:
        print(f"S3 not reachable ({type(error).__name__}) - using the local copy in data/s3_fallback")
        return LocalS3(DATA / "s3_fallback")


s3 = connect_s3()
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="2024/110")
for obj in response.get("Contents", [])[:5]:
    print(obj["Key"], obj["Size"])

`list_objects_v2` returns at most 1,000 keys per call; the real bucket has ~12,000 stations per year. For more, you page through
the results with `ContinuationToken` (homework).

### Exercise 3 (Core) — Land the weather files
For each station in `STATIONS`, download `2024/<station>.csv` into `LANDING / "noaa" / "2024"` — but only if it is not there yet.
Return the list of downloaded keys in `downloaded`. Run the cell twice: the second time nothing should be downloaded.

In [ ]:
target_dir = LANDING / "noaa" / "2024"
target_dir.mkdir(parents=True, exist_ok=True)

# YOUR CODE HERE

print(downloaded)

### Exercise 4 (Core) — Clean real-world weather data
NOAA's data is real and has real quirks: numbers padded with spaces, temperatures in **Fahrenheit**, and **sentinel values**
for missing data (`9999.9` for temperature, `99.99` for precipitation) instead of empty fields.

Build `weather` with columns `country`, `date` (date), `temp_c` (1 decimal), `rain_mm` (1 decimal; precipitation is in inches, 1 in = 25.4 mm)
and `_source_file`. Sentinels must become `null`. Then show, for **September 2024**, the average temperature, the number of rainy days
(`rain_mm > 0`) and the number of days **without** a rain measurement per country.

Hint: `F.trim(...).cast("double")`; °C = (°F − 32) × 5 / 9; map the station to a country with a small DataFrame made from `STATIONS` and a join.

In [ ]:
raw_weather = spark.read.csv(str(target_dir / "*.csv"), header=True).withColumn("_source_file", F.col("_metadata.file_name"))
raw_weather.select("STATION", "DATE", "NAME", "TEMP", "PRCP").show(3)

# YOUR CODE HERE

**Discuss:** in mid-September 2024, storm Boris brought record rain to Budapest. Does the table say so? Look at the last column.
**Missing is not zero** — if we had replaced the sentinel with `0`, our report would claim it never rained.

> **Under the hood:** on a real platform Spark reads S3 **directly**, in parallel, without downloading first:
> ```python
> spark.read.csv("s3a://noaa-gsod-pds/2024/")   # needs the hadoop-aws package and credentials or an IAM role
> ```
> Databricks, EMR and friends have this configured for you. Downloading first, as we did, is common for small partner files.

---
## Part 5 — Databases

Most business data lives in **production databases** (PostgreSQL, MySQL, SQL Server, Oracle): subscriptions, payments, customers.
Rules for data engineers:
- The database serves the app. **Never** run heavy queries on it at peak time; use a **read replica** if there is one.
- Use a **read-only** user.
- Pull **incrementally**: only rows changed since last time, using an `updated_at` column — a **watermark**.
- Always pass values as **parameters** (`?`), never paste them into the query text: that prevents **SQL injection**.

Wavely's billing database is simulated by a small SQLite file. (Writing SQL is covered in your SQL course; here we need one simple query.)

In [ ]:
import sqlite3
from contextlib import closing

DB_URI = f"file:{DATA / 'db' / 'wavely_billing.sqlite'}?mode=ro"  # mode=ro: read-only connection

with closing(sqlite3.connect(DB_URI, uri=True)) as con:
    columns = [row[1] for row in con.execute("PRAGMA table_info(subscriptions)")]
    sample = con.execute("SELECT * FROM subscriptions LIMIT 3").fetchall()
print(columns)
for row in sample:
    print(row)

### Exercise 5 (Core) — Incremental pull with a watermark
Write `pull_changed_subscriptions(since)` that returns a Spark DataFrame with all subscriptions where `updated_at > since`.
Use the query `"SELECT * FROM subscriptions WHERE updated_at > ?"` with `(since,)` as parameter, then `spark.createDataFrame(rows, SUBSCRIPTIONS_SCHEMA)`.
The explicit schema matters: when nothing changed, `rows` is empty and Spark could not guess the columns.

Then simulate two nightly runs: the first with watermark `"2026-09-10 00:00:00"`, the second with the **new** watermark = the largest
`updated_at` of the first result. How many rows does the second run return, and why?

In [ ]:
SUBSCRIPTIONS_SCHEMA = "user_id STRING, plan STRING, monthly_price_eur DOUBLE, status STRING, updated_at STRING"


def pull_changed_subscriptions(since):
    # YOUR CODE HERE
    ...


first = pull_changed_subscriptions("2026-09-10 00:00:00")
print(first.count(), "changed rows")  # expected: 57 changed rows
first.groupBy("plan", "status").count().show()

# YOUR CODE HERE

> **Under the hood:** with Spark you can read big tables straight from the database, in parallel:
> ```python
> spark.read.jdbc(url="jdbc:postgresql://replica:5432/billing", table="subscriptions",
>                 properties={"user": os.environ["DB_USER"], "password": os.environ["DB_PASSWORD"]})
> ```
> For very busy databases, companies use **change data capture (CDC)**: a tool like Debezium reads the database's change log and
> **pushes** every insert/update/delete into a stream. No queries on the database at all.

---
## Part 6 — Push sources (concepts)

| Mechanism | How it works | Example |
|---|---|---|
| upload to our storage | the partner gets write access to *our* S3 bucket/SFTP folder | a label uploads releases to `s3://wavely-landing/danube/` |
| webhook | the sender calls *our* HTTP endpoint when something happens | the payment provider tells us "payment succeeded" |
| message queue / stream | events are published to a topic; we consume them | every play goes into a Kafka topic in real time |
| CDC | database changes are pushed as events | subscription updates from PostgreSQL via Debezium |

Push sources often deliver **at least once** — the same message can arrive twice. Sound familiar? That's why our silver layer deduplicates by `event_id`.
Streaming with Spark (**Structured Streaming**) is a topic of the distributed processing course.

---
## Part 7 — Capstone kickoff

You will build your own small pipeline and present it in **session 10**. Choose one dataset (in `data/capstone/`):

| Option | Data | Difficulty |
|---|---|---|
| **A — Podcasts** | shows, episodes, 7 days of listen events (CSV) | getting started |
| **B — Concert tickets** | venues, concerts, one messy ticket sales export | medium: lots of cleaning |
| **C — Playlists** | 7 daily snapshots of nested playlist JSON | advanced: nested data + changes over time |

Open `notebooks/capstone_<option>.ipynb` for the brief and a starter structure. You can work alone or in pairs.

## Wrap-up

| Concept | Example |
|---|---|
| pull vs push | we fetch (SFTP, S3, DB, API) vs they deliver (uploads, webhooks, queues, CDC) |
| secrets | `os.environ`, secrets manager, Colab Secrets — never in code |
| SFTP | `paramiko`, `listdir_attr`, `get`, skip `.tmp`, host keys |
| S3 | bucket, key, prefix, `list_objects_v2`, `download_file`, IAM roles |
| databases | read-only, read replica, `updated_at` watermark, query parameters |
| landing | copy unchanged first, then bronze → silver → gold |

**Homework:** `homework/hw07_data_sourcing.ipynb` — and choose your capstone option.